# Notebook 4: 최종 평가

## 목표
SlideAudit 라벨 데이터로 전체 파이프라인 AUC 측정.  
Isolation Forest(기존) vs 새 파이프라인(CNN + HMM) 정량 비교.

## 평가 지표
- **AUC-ROC**: 이상 탐지 정확도 (FashionHarmony와 동일한 지표로 비교 가능)
- **Precision / Recall @ threshold**
- **역할 분류 정확도** (CNN)
- **구조 이상 탐지율** (HMM)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'
SLIDEAUDIT_DIR = f'{BASE_DIR}/slideaudit'

In [ ]:
!pip install -q timm hmmlearn datasets
!apt-get install -q libreoffice
print('설치 완료')

## 1. SlideAudit 데이터셋 로드

GitHub: https://github.com/WeichenSunAlex/UIST_2025  
2,400장 슬라이드, 폰트/색상/레이아웃 결함 라벨

In [ ]:
import os
import subprocess

os.makedirs(SLIDEAUDIT_DIR, exist_ok=True)

# SlideAudit 클론
if not os.path.exists(f'{SLIDEAUDIT_DIR}/UIST_2025'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/WeichenSunAlex/UIST_2025',
         f'{SLIDEAUDIT_DIR}/UIST_2025'],
        check=True
    )
    print('SlideAudit 클론 완료')
else:
    print('SlideAudit 이미 존재')

# 디렉토리 구조 확인
for root, dirs, files in os.walk(f'{SLIDEAUDIT_DIR}/UIST_2025'):
    depth = root.replace(f'{SLIDEAUDIT_DIR}/UIST_2025', '').count(os.sep)
    if depth < 2:
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth == 1:
            for f in files[:5]:
                print(f'{indent}  {f}')

In [ ]:
# SlideAudit 라벨 파일 로드 (구조 확인 후 경로 조정)
import pandas as pd
import numpy as np
import json
from pathlib import Path

# 라벨 파일 찾기
label_files = list(Path(f'{SLIDEAUDIT_DIR}/UIST_2025').rglob('*.json')) + \
              list(Path(f'{SLIDEAUDIT_DIR}/UIST_2025').rglob('*.csv'))

print('발견된 라벨 파일:')
for f in label_files:
    print(f'  {f}')

In [ ]:
# 라벨 파일 구조에 따라 아래 코드를 조정
# SlideAudit 라벨: 폰트/색상/레이아웃/콘텐츠 결함 multi-label

# 예시 구조 (실제 파일 확인 후 수정):
# {
#   "slide_id": "xxx",
#   "image_path": "xxx.png",
#   "has_defect": true,
#   "defect_types": ["typography", "color"]
# }

# 라벨 파일 로드 (경로는 위 출력 확인 후 수정)
label_file = label_files[0]  # 첫 번째 파일 시도

if label_file.suffix == '.json':
    with open(label_file) as f:
        raw = json.load(f)
    print(f'JSON 구조 (첫 항목):')
    if isinstance(raw, list):
        print(json.dumps(raw[0], indent=2, ensure_ascii=False))
        slideaudit_df = pd.DataFrame(raw)
    else:
        print(json.dumps(list(raw.items())[:3], indent=2, ensure_ascii=False))
elif label_file.suffix == '.csv':
    slideaudit_df = pd.read_csv(label_file)
    print(slideaudit_df.head())
    print(slideaudit_df.columns.tolist())

print(f'\nSlideAudit 샘플 수: {len(slideaudit_df)}')

## 2. 파이프라인 구성

In [ ]:
import torch
import timm
import torch.nn as nn
import pickle
from torchvision import transforms
from PIL import Image
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
NUM_CLASSES = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# CNN 모델 재정의 (Notebook 2와 동일)
class SlideRoleClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=False,
                                          num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat_dim, 256),
            nn.ReLU(), nn.Dropout(dropout * 0.5), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def extract_features(self, x):
        return self.backbone(x)


# 모델 로드
cnn_model = SlideRoleClassifier().to(device)
ckpt = torch.load(f'{MODELS_DIR}/role_classifier_best.pt', map_location=device)
cnn_model.load_state_dict(ckpt['model_state_dict'])
cnn_model.eval()
print(f'CNN 모델 로드 완료 (val_acc={ckpt["val_acc"]:.4f})')

with open(f'{MODELS_DIR}/hmm_model.pkl', 'rb') as f:
    hmm_model = pickle.load(f)
print('HMM 모델 로드 완료')

with open(f'{MODELS_DIR}/hmm_thresholds.json') as f:
    thresholds = json.load(f)
print(f'임계값 로드: {thresholds}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 3. 슬라이드별 이상 점수 계산

최종 이상 점수 = **0.5 × IF이상점수** + **0.3 × HMM이상점수** + **0.2 × 역할불일치점수**

In [ ]:
from tqdm.notebook import tqdm


def compute_anomaly_score(image_paths: list[str]) -> dict:
    """
    한 덱의 슬라이드 이미지 리스트를 받아 이상 점수 반환.
    Returns:
        slide_scores: 슬라이드별 이상 점수 (0~1)
        deck_structural_score: 덱 전체의 구조 이상 점수 (0~1)
    """
    if not image_paths:
        return {'slide_scores': [], 'deck_structural_score': 0.5}

    # 1. CNN 임베딩 + 역할 예측
    embeddings = []
    pred_roles = []

    with torch.no_grad():
        for img_path in image_paths:
            img = Image.open(img_path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            emb = cnn_model.extract_features(tensor).squeeze(0).cpu().numpy()
            logits = cnn_model.classifier(torch.tensor(emb).unsqueeze(0).to(device))
            role = logits.argmax(dim=1).item()
            embeddings.append(emb)
            pred_roles.append(role)

    embeddings_np = np.array(embeddings)  # (N, 1536)

    # 2. 역할별 Isolation Forest (덱 내)
    if len(embeddings_np) >= 3:
        iso = IsolationForest(n_estimators=100, contamination=0.2, random_state=42)
        iso.fit(embeddings_np)
        raw_scores = iso.decision_function(embeddings_np)
        # 정규화 후 반전 (높을수록 이상)
        s_min, s_max = raw_scores.min(), raw_scores.max()
        if_scores = 1.0 - (raw_scores - s_min) / (s_max - s_min + 1e-8)
    else:
        if_scores = np.full(len(embeddings_np), 0.5)

    # 3. HMM 구조 점수 (덱 전체)
    seq = np.array(pred_roles).reshape(-1, 1)
    log_likelihood = hmm_model.score(seq) / len(seq)
    mean_ll = thresholds['mean']
    std_ll = thresholds['std']
    # log-likelihood가 낮을수록 이상 → 0~1 정규화
    z_score = (mean_ll - log_likelihood) / (std_ll + 1e-8)
    hmm_score = float(np.clip(z_score / 3.0, 0, 1))  # 3-sigma 기준

    # 4. 최종 점수 합산
    final_scores = 0.7 * if_scores + 0.3 * hmm_score

    return {
        'slide_scores': final_scores.tolist(),
        'pred_roles': pred_roles,
        'deck_structural_score': hmm_score,
        'if_scores': if_scores.tolist(),
    }


print('이상 점수 계산 함수 준비 완료')

## 4. SlideAudit로 AUC 측정

In [ ]:
# SlideAudit 이미지 경로와 라벨 매핑
# 아래 코드는 SlideAudit 실제 구조에 맞게 수정 필요
# has_defect 컬럼: True = 이상, False = 정상

all_scores = []
all_labels = []

# 덱 단위로 이미지 그룹핑 (SlideAudit 구조에 따라 수정)
# slideaudit_df에 'deck_id', 'image_path', 'has_defect' 컬럼이 있다고 가정

for deck_id, group in tqdm(slideaudit_df.groupby('deck_id'), desc='AUC 계산'):
    group = group.sort_values('slide_idx') if 'slide_idx' in group.columns else group
    image_paths = group['image_path'].tolist()
    labels = group['has_defect'].astype(int).tolist()

    # 이미지 파일 존재 확인
    valid = [(p, l) for p, l in zip(image_paths, labels) if Path(p).exists()]
    if not valid:
        continue
    valid_paths, valid_labels = zip(*valid)

    result = compute_anomaly_score(list(valid_paths))
    all_scores.extend(result['slide_scores'])
    all_labels.extend(valid_labels)

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

print(f'평가 슬라이드 수: {len(all_scores)}')
print(f'이상 슬라이드 비율: {all_labels.mean():.2%}')

In [ ]:
# 기존 Isolation Forest (베이스라인)
from sklearn.ensemble import IsolationForest as IF_baseline

embeddings_all = []
for img_path in tqdm(slideaudit_df['image_path'].tolist()[:len(all_scores)], desc='베이스라인 임베딩'):
    if not Path(img_path).exists():
        embeddings_all.append(np.zeros(1536))
        continue
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = cnn_model.extract_features(tensor).squeeze(0).cpu().numpy()
    embeddings_all.append(emb)

emb_matrix = np.array(embeddings_all)
iso_baseline = IF_baseline(n_estimators=100, contamination=0.2, random_state=42)
iso_baseline.fit(emb_matrix)
baseline_raw = iso_baseline.decision_function(emb_matrix)
s_min, s_max = baseline_raw.min(), baseline_raw.max()
baseline_scores = 1.0 - (baseline_raw - s_min) / (s_max - s_min + 1e-8)

# AUC 계산
auc_new = roc_auc_score(all_labels, all_scores)
auc_baseline = roc_auc_score(all_labels, baseline_scores[:len(all_labels)])

print(f'\n=== 결과 비교 ===')
print(f'기존 Isolation Forest AUC: {auc_baseline:.4f}')
print(f'새 파이프라인 AUC:          {auc_new:.4f}')
print(f'개선폭:                    +{auc_new - auc_baseline:.4f}')

In [ ]:
# ROC 커브 비교
from sklearn.metrics import roc_curve

fpr_new, tpr_new, _ = roc_curve(all_labels, all_scores)
fpr_base, tpr_base, _ = roc_curve(all_labels, baseline_scores[:len(all_labels)])

plt.figure(figsize=(8, 6))
plt.plot(fpr_new, tpr_new, color='steelblue',
         label=f'새 파이프라인 (AUC={auc_new:.4f})', linewidth=2)
plt.plot(fpr_base, tpr_base, color='tomato', linestyle='--',
         label=f'Isolation Forest (AUC={auc_baseline:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k:', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve 비교')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/roc_comparison.png', dpi=120)
plt.show()
print('ROC 커브 저장 완료')

In [ ]:
# 최종 결과 저장
final_results = {
    'baseline_auc': float(auc_baseline),
    'new_pipeline_auc': float(auc_new),
    'improvement': float(auc_new - auc_baseline),
    'n_eval_slides': int(len(all_labels)),
    'defect_ratio': float(all_labels.mean()),
}

with open(f'{MODELS_DIR}/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print('=== Notebook 4 완료 ===')
print(json.dumps(final_results, indent=2))
print('\n모든 모델 파일이 Google Drive에 저장되었습니다.')
print('다음 단계: 백엔드 통합 (backend/app/pipeline/ 교체)')